# Agent Handoffs with the Mistral Agents API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mistralai/cookbook/blob/main/mistral/agents/agents_api/agents_handoffs.ipynb)

This notebook demonstrates how to build multi-agent systems using Mistral's Agents API with automatic and manual handoff capabilities. Handoffs allow a **triage agent** to delegate tasks to **specialized agents**, enabling modular and scalable agent architectures.

**Author**: [abdelhadi703](https://github.com/abdelhadi703)

## Setup

Install the Mistral AI SDK and set up the client.

In [ ]:
!pip install -q "mistralai>=1.6.0"

In [ ]:
import os
from getpass import getpass
from mistralai import Mistral

api_key = os.environ.get("MISTRAL_API_KEY") or getpass("Enter your Mistral API key: ")
client = Mistral(api_key=api_key)

## 1. Concepts: What Are Agent Handoffs?

A **handoff** lets one agent delegate a task to another agent. This is useful when:

- Different agents specialize in different domains (coding, research, customer support).
- A **triage agent** acts as a router, analyzing the user's query and forwarding it to the most relevant specialist.
- You want to build a **multi-agent network** where agents can call each other.

Mistral supports two handoff execution modes:

| Mode | Parameter | Behavior |
|------|-----------|----------|
| **Server** (default) | `handoff_execution="server"` | The server automatically routes the conversation to the target agent and returns the final response. |
| **Client** | `handoff_execution="client"` | The server returns a handoff event, and your code decides how to proceed. |

## 2. Creating Agents with Handoffs

Let's create three agents:
- A **Code Expert** that uses the code interpreter tool.
- A **Research Expert** that uses web search.
- A **Triage Agent** that routes queries to the right specialist.

In [ ]:
# Agent specialized in coding tasks
code_agent = client.beta.agents.create(
    model="mistral-medium-latest",
    name="Code Expert",
    description="Agent specialized in writing, reviewing, and debugging code.",
    instructions="You are a coding expert. Write clean, well-commented code. Use the code interpreter to run and validate your solutions.",
    tools=[{"type": "code_interpreter"}],
)
print(f"Code Expert agent created: {code_agent.id}")

# Agent specialized in web research
research_agent = client.beta.agents.create(
    model="mistral-medium-latest",
    name="Research Expert",
    description="Agent specialized in searching the web for up-to-date information.",
    instructions="You are a research expert. Use web search to find accurate, recent information. Always cite your sources.",
    tools=[{"type": "web_search"}],
)
print(f"Research Expert agent created: {research_agent.id}")

# Triage agent that routes to specialists
triage_agent = client.beta.agents.create(
    model="mistral-medium-latest",
    name="Triage Agent",
    description="Agent that routes user queries to the appropriate specialist.",
    instructions=(
        "You are a triage agent. Analyze the user's query and route it to the right specialist:\n"
        "- For coding questions (write code, debug, explain code), hand off to Code Expert.\n"
        "- For research questions (current events, factual lookups), hand off to Research Expert.\n"
        "- For general questions, answer directly."
    ),
    handoffs=[code_agent.id, research_agent.id],
)
print(f"Triage Agent created: {triage_agent.id}")

## 3. Server Mode: Automatic Handoff Execution

In **server mode** (`handoff_execution="server"`, the default), the Mistral server automatically executes the handoff. The triage agent decides which specialist to call, and the server handles the routing transparently.

We use `start_stream` to see the handoff events in real time.

In [ ]:
from mistralai import (
    AgentHandoffStartedEvent,
    AgentHandoffDoneEvent,
    MessageOutputEvent,
    ToolExecutionStartedEvent,
    ToolExecutionDoneEvent,
    ResponseStartedEvent,
    ResponseDoneEvent,
)


def run_agent_stream(agent_id, user_input, handoff_execution="server"):
    """Run an agent with streaming and display handoff events."""
    print(f"User: {user_input}\n")
    conversation_id = None

    response = client.beta.conversations.start_stream(
        agent_id=agent_id,
        inputs=user_input,
        handoff_execution=handoff_execution,
    )

    with response as event_stream:
        for event in event_stream:
            match event.data:
                case ResponseStartedEvent():
                    conversation_id = event.data.conversation_id
                case AgentHandoffStartedEvent():
                    print(f"\n>> Handoff started (leaving: {event.data.previous_agent_name})")
                case AgentHandoffDoneEvent():
                    print(f">> Handoff done (entering: {event.data.next_agent_name})\n")
                case ToolExecutionStartedEvent():
                    print(f"  [Tool: {event.data.name}]")
                case MessageOutputEvent():
                    if isinstance(event.data.content, str):
                        print(event.data.content, end="")

    print("\n")
    return conversation_id

In [ ]:
# This query should be routed to the Code Expert
conv_id = run_agent_stream(
    triage_agent.id,
    "Write a Python function to calculate the nth Fibonacci number using memoization.",
)

In [ ]:
# This query should be routed to the Research Expert
conv_id_research = run_agent_stream(
    triage_agent.id,
    "What were the key announcements at the latest Mistral AI product launch?",
)

## 4. Client Mode: Manual Handoff Control

In **client mode** (`handoff_execution="client"`), the server returns a handoff event instead of automatically routing. Your application code then decides how to handle it — useful for adding custom logic, logging, or approval steps before the handoff proceeds.

In [ ]:
print("User: Explain how quicksort works and find its current Wikipedia page.\n")

response = client.beta.conversations.start_stream(
    agent_id=triage_agent.id,
    inputs="Explain how quicksort works and find its current Wikipedia page.",
    handoff_execution="client",
)

conversation_id = None
handoff_target_id = None
handoff_target_name = None

with response as event_stream:
    for event in event_stream:
        match event.data:
            case ResponseStartedEvent():
                conversation_id = event.data.conversation_id
            case AgentHandoffStartedEvent():
                print(f">> Handoff requested (leaving: {event.data.previous_agent_name})")
            case AgentHandoffDoneEvent():
                handoff_target_id = event.data.next_agent_id
                handoff_target_name = event.data.next_agent_name
                print(f">> Target agent: {handoff_target_name} ({handoff_target_id})")
            case MessageOutputEvent():
                if isinstance(event.data.content, str):
                    print(event.data.content, end="")

print(f"\n\nConversation ID: {conversation_id}")
print(f"Handoff target: {handoff_target_name}")

In [ ]:
# Now we manually continue the conversation with the target agent
if handoff_target_id and conversation_id:
    print(f"Continuing conversation with {handoff_target_name}...\n")

    response = client.beta.conversations.append_stream(
        conversation_id=conversation_id,
        inputs="Please proceed with the task.",
        handoff_execution="server",
    )

    with response as event_stream:
        for event in event_stream:
            match event.data:
                case AgentHandoffDoneEvent():
                    print(f"\n>> Handed off to: {event.data.next_agent_name}\n")
                case ToolExecutionStartedEvent():
                    print(f"  [Tool: {event.data.name}]")
                case MessageOutputEvent():
                    if isinstance(event.data.content, str):
                        print(event.data.content, end="")
    print()

## 5. Multi-Agent Network: Travel Assistant Pattern

A more advanced pattern involves creating a **network of agents** where each specialist can hand off to others. This mirrors the `travel_assistant` example in the cookbook.

We'll create:
- A **Travel Triage** agent that routes user queries.
- A **Flights** agent for flight-related queries.
- A **Hotels** agent for accommodation queries.
- An **Activities** agent for local activities and events.

In [ ]:
# Create specialized travel agents
flights_agent = client.beta.agents.create(
    model="mistral-medium-latest",
    name="Flights Agent",
    description="Agent specialized in finding and recommending flights.",
    instructions=(
        "You are a flights specialist. Help users find flights, compare options, "
        "and provide booking advice. Use web search to find current flight information."
    ),
    tools=[{"type": "web_search"}],
)

hotels_agent = client.beta.agents.create(
    model="mistral-medium-latest",
    name="Hotels Agent",
    description="Agent specialized in finding and recommending hotels.",
    instructions=(
        "You are a hotels specialist. Help users find accommodations, compare hotels, "
        "and provide booking advice. Use web search to find current hotel information."
    ),
    tools=[{"type": "web_search"}],
)

activities_agent = client.beta.agents.create(
    model="mistral-medium-latest",
    name="Activities Agent",
    description="Agent specialized in finding local activities, tours, and events.",
    instructions=(
        "You are an activities and events specialist. Help users discover things to do, "
        "local tours, restaurants, and cultural events. Use web search to find current information."
    ),
    tools=[{"type": "web_search"}],
)

# Create the travel triage agent
travel_triage = client.beta.agents.create(
    model="mistral-medium-latest",
    name="Travel Triage",
    description="Main travel assistant that routes queries to specialists.",
    instructions=(
        "You are a travel assistant. Route user queries to the right specialist:\n"
        "- Flight questions -> Flights Agent\n"
        "- Hotel/accommodation questions -> Hotels Agent\n"
        "- Activities/events/restaurants -> Activities Agent\n"
        "- General travel questions -> answer directly."
    ),
    handoffs=[flights_agent.id, hotels_agent.id, activities_agent.id],
)

print("Travel agents created!")
print(f"  Travel Triage: {travel_triage.id}")
print(f"  Flights Agent: {flights_agent.id}")
print(f"  Hotels Agent:  {hotels_agent.id}")
print(f"  Activities Agent: {activities_agent.id}")

In [ ]:
# Set up cross-handoffs so specialists can route to each other
client.beta.agents.update(agent_id=flights_agent.id, handoffs=[hotels_agent.id, activities_agent.id])
client.beta.agents.update(agent_id=hotels_agent.id, handoffs=[flights_agent.id, activities_agent.id])
client.beta.agents.update(agent_id=activities_agent.id, handoffs=[flights_agent.id, hotels_agent.id])

print("Cross-handoffs configured!")

In [ ]:
# Test the travel network
conv_travel = run_agent_stream(
    travel_triage.id,
    "I'm planning a trip to Tokyo next month. Can you find some popular activities and local food tours?",
)

## 6. Inspecting Conversation History

After a conversation with handoffs, you can use `get_history()` to inspect the full sequence of events, including `AgentHandoffEntry` entries that show which agents were involved.

In [ ]:
from mistralai import AgentHandoffEntry

if conv_travel:
    history = client.beta.conversations.get_history(conversation_id=conv_travel)

    print(f"Conversation {conv_travel}")
    print(f"Total entries: {len(history.entries)}\n")

    for i, entry in enumerate(history.entries):
        entry_type = entry.type if hasattr(entry, "type") else type(entry).__name__
        print(f"Entry {i}: type={entry_type}")

        if isinstance(entry, AgentHandoffEntry):
            print(f"  Handoff: {entry.previous_agent_name} -> {entry.next_agent_name}")
        elif hasattr(entry, "role"):
            content_preview = ""
            if hasattr(entry, "content") and isinstance(entry.content, str):
                content_preview = entry.content[:100] + "..." if len(entry.content) > 100 else entry.content
            elif hasattr(entry, "content") and isinstance(entry.content, list):
                for chunk in entry.content:
                    if hasattr(chunk, "text"):
                        content_preview = chunk.text[:100] + "..."
                        break
            print(f"  Role: {entry.role}, Content: {content_preview}")
        print()

## 7. Cleanup

Delete all agents created during this notebook to keep your account tidy.

In [ ]:
agents_to_delete = [
    ("Triage Agent", triage_agent),
    ("Code Expert", code_agent),
    ("Research Expert", research_agent),
    ("Travel Triage", travel_triage),
    ("Flights Agent", flights_agent),
    ("Hotels Agent", hotels_agent),
    ("Activities Agent", activities_agent),
]

for name, agent in agents_to_delete:
    try:
        client.beta.agents.delete(agent_id=agent.id)
        print(f"Deleted {name} ({agent.id})")
    except Exception as e:
        print(f"Failed to delete {name}: {e}")

print("\nCleanup complete!")

## Best Practices

1. **Write clear `description` fields.** The triage agent uses descriptions to decide which specialist to route to. Be specific about each agent's capabilities.

2. **Use `instructions` to define behavior.** Instructions act as the system prompt. Include guidance on when to answer directly vs. when to hand off.

3. **Use `handoffs` at creation time or via `update`.** You can pass `handoffs` directly to `agents.create()`, or configure them later with `agents.update()`. The latter is necessary for circular handoff networks.

4. **Choose the right execution mode:**
   - **Server mode** is simpler and works well for straightforward routing.
   - **Client mode** gives you control over the routing decision, useful for logging, approval workflows, or custom routing logic.

5. **Clean up agents.** Agents persist on your account. Always delete them when they are no longer needed.

6. **Inspect history for debugging.** Use `get_history()` to see the full conversation flow, including which agents handled which parts.